# 08 - Spatially Variable Genes (Moran's I)

## Learning objectives
1. Build the spatial neighbor graph and compute spatial autocorrelation (Moran's I).
2. Identify genes with spatial structure (SVGs) and visualize them.
3. Explain how SVGs differ from highly variable genes (HVGs).

## Concept
**HVGs** are genes with high variance across spots - but variance ignores *position*. A
gene could be high-variance yet scattered randomly. **SVGs** are genes whose expression is
**spatially organized**: nearby spots have similar values. **Moran's I** measures exactly
that (~ -1..+1; high positive = strong spatial pattern). Think of it as detecting genes
that form coherent 'regions' rather than salt-and-pepper noise.


In [ ]:
# --- Standard setup: make `utils` importable and seed RNGs ---
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'utils').exists():
    ROOT = ROOT.parent  # in case the notebook is opened from a subfolder
sys.path.insert(0, str(ROOT))

from utils import st_helpers as st
st.set_seeds()  # reproducibility (seed = 0)
print('Project root:', st.project_root())


In [ ]:
import scanpy as sc
import squidpy as sq

adata = st.load_adata('adata_clustered.h5ad')
# Ensure the spatial graph exists (created in nb 07, but be self-contained).
if 'spatial_connectivities' not in adata.obsp:
    sq.gr.spatial_neighbors(adata, coord_type='grid', n_neighs=6)
adata


### Compute Moran's I
We test the highly variable genes (computing on all ~18k genes is slow and mostly noise).
Results land in `adata.uns['moranI']`, sorted by the statistic.

In [ ]:
hvg = adata.var_names[adata.var['highly_variable']].tolist() if 'highly_variable' in adata.var else adata.var_names[:2000].tolist()
print('Testing', len(hvg), 'genes for spatial autocorrelation...')
sq.gr.spatial_autocorr(adata, mode='moran', genes=hvg, n_perms=100,
                       n_jobs=1, seed=st.SEED)
moran = adata.uns['moranI']
moran.head(15)


**Expected output:** a table indexed by gene with an `I` column (and a p-value), sorted
descending. Top genes have `I` well above 0 - strong spatial structure.

### Visualize the top SVGs over tissue

In [ ]:
top_svgs = st.genes_present(adata, moran.head(6).index.tolist())
print('Top SVGs:', top_svgs)
sq.pl.spatial_scatter(adata, color=top_svgs, ncols=3, size=1.3, cmap='magma')


### SVGs vs HVGs - how much do they overlap?
A gene can be highly variable but not spatially structured, and vice versa. Let's compare.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Rank genes by HVG dispersion and by Moran's I, then look at the relationship.
common = [g for g in moran.index if g in adata.var_names]
I_vals = moran.loc[common, 'I']
disp = adata.var.loc[common, 'dispersions_norm'] if 'dispersions_norm' in adata.var else None

if disp is not None:
    plt.figure(figsize=(6, 5))
    plt.scatter(disp, I_vals, s=8, alpha=0.4)
    plt.xlabel('HVG normalized dispersion (variance-based)')
    plt.ylabel("Moran's I (spatial structure)")
    plt.title('Highly variable is not the same as spatially variable')
    plt.axhline(0, color='gray', lw=0.8)
    plt.show()
    print('Correlation between dispersion and Moran I:',
          round(float(np.corrcoef(disp, I_vals)[0, 1]), 3))


**Expected output:** a weak/moderate positive but clearly scattered relationship - many
high-dispersion genes have modest Moran's I, confirming the two notions are related but
**not** the same.

## Common pitfalls
- Forgetting `sq.gr.spatial_neighbors` first - Moran's I needs the spatial graph.
- Testing all genes - slow and dominated by noise; restrict to HVGs.
- Reading Moran's I as effect size - it measures *pattern*, not magnitude of expression.

## Interpretation
SVGs are the genes that 'draw the map' - those whose expression respects tissue geography.
They are the natural targets for spatial-domain interpretation.

## What this means biologically
Spatially structured genes typically mark anatomical compartments or gradients (e.g.
cortical-layer or region-specific programs). They are prime candidates for understanding
how function is organized across the tissue - and, in disease, how niches differ.

---
**Next:** `09_image_feature_extraction_from_histology.ipynb`.
